# Replace 1

### Below code 

```python
from kernels import get_kernel
cap = torch.cuda.get_device_capability()

# varunneal's FA3 is Hopper only, use kernels-community on non-Hopper GPUs
repo = "varunneal/flash-attention-3" if cap == (9, 0) else "kernels-community/flash-attn3"
fa3 = get_kernel(repo).flash_attn_interface
```

## with

```python
fa3 = None  # Replace above lines with this
```

-----

# Replace 2


## below line

```python
y = fa3.flash_attn_func(q, k, v, causal=True, window_size=window_size)
```

## with

```python
# convert to (B, heads, T, head_dim)
q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)

# PyTorch attention (fallback)
y = torch.nn.functional.scaled_dot_product_attention(
    q, k, v,
    attn_mask=None,
    is_causal=True
)

# back to (B, T, C)
y = y.transpose(1, 2).contiguous().view(B, T, -1)
```

-----

# Replace 3

## below code 

```python
# ============================================================================================================
# Replace this function
# ============================================================================================================
def setup_optimizer(self, unembedding_lr=0.004, embedding_lr=0.2, matrix_lr=0.02,
                    weight_decay=0.0, adam_betas=(0.8, 0.95), scalar_lr=0.5):
    model_dim = self.config.n_embd
    matrix_params = list(self.transformer.h.parameters())
    value_embeds_params = list(self.value_embeds.parameters())
    embedding_params = list(self.transformer.wte.parameters())
    lm_head_params = list(self.lm_head.parameters())
    resid_params = [self.resid_lambdas]
    x0_params = [self.x0_lambdas]
    assert len(list(self.parameters())) == (len(matrix_params) + len(embedding_params) +
        len(lm_head_params) + len(value_embeds_params) + len(resid_params) + len(x0_params))
    # Scale LR ∝ 1/√dmodel (tuned at 768 dim)
    dmodel_lr_scale = (model_dim / 768) ** -0.5
    print(f"Scaling AdamW LRs by 1/sqrt({model_dim}/768) = {dmodel_lr_scale:.6f}")
    param_groups = [
        dict(kind='adamw', params=lm_head_params, lr=unembedding_lr * dmodel_lr_scale, betas=adam_betas, eps=1e-10, weight_decay=0.0),
        dict(kind='adamw', params=embedding_params, lr=embedding_lr * dmodel_lr_scale, betas=adam_betas, eps=1e-10, weight_decay=0.0),
        dict(kind='adamw', params=value_embeds_params, lr=embedding_lr * dmodel_lr_scale, betas=adam_betas, eps=1e-10, weight_decay=0.0),
        dict(kind='adamw', params=resid_params, lr=scalar_lr * 0.01, betas=adam_betas, eps=1e-10, weight_decay=0.0),
        dict(kind='adamw', params=x0_params, lr=scalar_lr, betas=(0.96, 0.95), eps=1e-10, weight_decay=0.0),
    ]
    for shape in sorted({p.shape for p in matrix_params}):
        group_params = [p for p in matrix_params if p.shape == shape]
        param_groups.append(dict(
            kind='muon', params=group_params, lr=matrix_lr,
            momentum=0.95, ns_steps=5, beta2=0.95, weight_decay=weight_decay,
        ))
    optimizer = MuonAdamW(param_groups)
    for group in optimizer.param_groups:
        group["initial_lr"] = group["lr"]
    return optimizer

```


### with below
```python
def setup_optimizer(self, *args, **kwargs):
    print("Using standard AdamW optimizer (Triton disabled)")

    optimizer = torch.optim.AdamW(
        self.parameters(),
        lr=3e-4,
        betas=(0.9, 0.95),
        weight_decay=0.01
        )
    return optimizer

```

---

# Comment below decoratores

```python
# ------------------------comment---------------------------------------------------
@torch.compile(dynamic=False, fullgraph=True)
# -----------------------------------------------------------------------------------

```


```python
# ------------------------comment---------------------------------------------------
@torch.compile(dynamic=False, fullgraph=True)
# ------------------------comment---------------------------------------------------
```



-----

# Change in model architecture

```python
# Model architecture
ASPECT_RATIO = 64       # Nothing Changed

# Optimization
TOTAL_BATCH_SIZE = 4*2048*2 # 32768 tokens per optimizer step # calculate it and perform the changes

# Model size reduce it according to your GPU
DEPTH = 6               # number of transformer layers # changes from 8 to 6 
DEVICE_BATCH_SIZE = 8  # per-device batch size (reduce if OOM) # from 128 to 8

```



----

# Disable 

```python
# ---------------------------------------------------------------------------
model = torch.compile(model, dynamic=False) # diable this line
# ---------------------------------------------------------------------------
```

---

# Replace

### below code
```python
for group in optimizer.param_groups:
    group["lr"] = group["initial_lr"] * lrm
    if group['kind'] == 'muon':
        group["momentum"] = muon_momentum
        group["weight_decay"] = muon_weight_decay
```

### with 
```python
for group in optimizer.param_groups:
    group["lr"] = 3e-4 * lrm
```

----

# Added Custome code for my problem statement

```python
#====================================================

print("\n--- TESTING MODEL ---\n")

model.eval()

prompt = """Classify the following text:
We are investing in renewable energy
Answer:"""

# Encode input
tokens = tokenizer.encode(prompt)
x = torch.tensor([tokens], device="cuda")

# Generate tokens
max_new_tokens = 20

for _ in range(max_new_tokens):
    with torch.no_grad():
        logits = model(x)
    
    next_token = torch.argmax(logits[0, -1]).unsqueeze(0)
    x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

# Decode output
output = tokenizer.decode(x[0].tolist())

print(output)

```



---